# 🌐 외부 API + Pydantic 정형 반환

 가짜 dict 데이터 → Pydantic 객체 반환  
 진짜 외부 API → 같은 Pydantic 객체 반환  

```python
(가짜 데이터, Pydantic 반환)
@tool
def get_exchange_rate(...) -> ExchangeRateResult:
    rates = {("USD","KRW"): 1300.0, ...}
    return ExchangeRateResult(...)

# 심화 (진짜 API, 같은 Pydantic 반환)
@tool
def get_exchange_rate(...) -> ExchangeRateInfo:
    data = requests.get("https://open.er-api.com/...").json()
    return ExchangeRateInfo(...)
```

**도구 안의 데이터 소스만 바뀌고** 정형 반환 패턴은 그대로. LLM 입장에서도 동일.

## 📋 사용할 외부 API 4가지

모두 **무료**이고 **인증 불필요**.

| API | 도구 반환 스키마 | 어떤 정보 |
| --- | --- | --- |
| ExchangeRate-API | `ExchangeRateInfo` | 실시간 환율 (161개국) |
| Wikipedia REST | `WikiInfo` | 한국어 위키 요약 |
| Open-Meteo | `WeatherInfo` | 전 세계 도시 날씨 |
| REST Countries | `CountryInfo` | 국가 정보 |

## 📋 노트북 순서

```
Step 0. 환경 설정 + 헬퍼 함수
Step 1. 가짜 API vs 진짜 API 직접 비교
Step 2. ① 환율 API → ExchangeRateInfo
Step 3. ② 위키피디아 검색 → WikiInfo
Step 4. ③ 날씨 API → WeatherInfo
Step 5. ④ 국가 정보 API → CountryInfo
Step 6. ⭐ 통합 미니 챗봇: 여행 도우미
Step 7. 도전 과제
```

> 💡 `run_with_tools` 헬퍼 함수를 그대로 가져와서 씁니다. 도구 정형 반환 패턴도 본 노트북에서 이어집니다.




---

# Step 0. 환경 설정 + 헬퍼 함수

`requests` 패키지를 추가로 설치합니다. 외부 API 호출에 사용.


In [1]:
# !uv add langchain langchain-openai python-dotenv pydantic requests

## 공통 import + 헬퍼 함수

In [3]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
import requests

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def run_with_tools(question: str, tools: list, llm_obj=llm):
    """4교시 본의 헬퍼 함수 — 4단계 흐름 자동 처리"""
    llm_with_tools = llm_obj.bind_tools(tools)
    messages = [HumanMessage(content=question)]

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        return response.content

    tool_map = {t.name: t for t in tools}
    for tc in response.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return llm_with_tools.invoke(messages).content

print("✅ 헬퍼 함수 준비 완료")

---

# Step 1. 가짜 API vs 진짜 API 직접 비교

4교시 본에서 만든 가짜 환율 도구의 한계를 직접 봅시다.

## 1-1. 가짜 환율 도구


In [3]:
# 4교시 본의 가짜 환율 도구 (Pydantic 반환은 그대로 유지)
class FakeRate(BaseModel):
    from_currency: str
    to_currency: str
    rate: float

@tool
def get_exchange_rate_fake(from_currency: str, to_currency: str) -> FakeRate:
    """환율 조회 (가짜 — 미리 정한 값만 반환)"""
    rates = {("USD","KRW"): 1300.0, ("EUR","KRW"): 1400.0}
    rate = rates.get((from_currency, to_currency), 0)
    return FakeRate(from_currency=from_currency, to_currency=to_currency, rate=rate)

# 한계 1: 미리 정한 통화쌍만 동작
print(get_exchange_rate_fake.invoke({"from_currency": "USD", "to_currency": "KRW"}))
print(get_exchange_rate_fake.invoke({"from_currency": "JPY", "to_currency": "KRW"}))
print("       ↑ 정의 안 한 통화쌍은 rate=0")

from_currency='USD' to_currency='KRW' rate=1300.0
from_currency='JPY' to_currency='KRW' rate=0.0
       ↑ 정의 안 한 통화쌍은 rate=0


**가짜 도구의 한계 3가지**

| 한계 | 설명 |
| --- | --- |
| 1. 데이터가 고정 | 환율이 매일 바뀌는데 코드에 박혀 있음 |
| 2. 범위 제한 | 정의한 통화쌍만 동작 |
| 3. 실무 가치 없음 | 실제 시스템에 못 씀 |

→ 진짜 API를 호출하는 도구로 바꿔봅시다. **Pydantic 반환 패턴은 그대로**.

## 1-2. 진짜 환율 API 미리보기


In [4]:
import requests
# 진짜 환율 API 직접 호출
url = "https://open.er-api.com/v6/latest/USD"
data = requests.get(url, timeout=10).json()

print(f"📅 기준일: {data['time_last_update_utc'][:25]}")
print(f"💱 1달러 = {data['rates']['KRW']:.2f}원")
print(f"💱 1달러 = {data['rates']['JPY']:.2f}엔")
print(f"💱 1달러 = {data['rates']['EUR']:.4f}유로")
print(f"\n총 {len(data['rates'])}개 통화 데이터")

📅 기준일: Tue, 15 Sep 2026 00:02:31
💱 1달러 = 1347.20원
💱 1달러 = 154.39엔
💱 1달러 = 0.8657유로

총 166개 통화 데이터


**👀 결과 관찰**

- 실시간 환율 데이터 (매 시간 업데이트)
- 161개국 통화 지원
- **인증 불필요, 무료**

이걸 그대로 Pydantic 도구로 만들면 됩니다.


---

# Step 2. ① 환율 API → `ExchangeRateInfo`

API 호출 코드를 `@tool` 데코레이터로 감싸고, **반환값을 Pydantic 객체로**.

## 2-1. Pydantic 스키마 + 도구 정의


In [5]:
# 정형 반환 스키마 — 본 노트북의 ExchangeRateResult와 같은 패턴
class ExchangeRateInfo(BaseModel):
    from_currency: str = Field(description="출발 통화 ISO 코드")
    to_currency: str = Field(description="도착 통화 ISO 코드")
    rate: float = Field(description="환율")
    last_updated: Optional[str] = Field(default=None, description="API 기준 시각")
    error: Optional[str] = Field(default=None, description="오류 메시지")

print("✅ ExchangeRateInfo 스키마 준비")

✅ ExchangeRateInfo 스키마 준비


In [6]:
@tool
def get_exchange_rate(from_currency: str, to_currency: str) -> ExchangeRateInfo:
    """실시간 환율을 조회합니다 (ExchangeRate-API).

    Args:
        from_currency: 출발 통화 ISO 코드 (예: 'USD', 'EUR', 'JPY', 'KRW')
        to_currency: 도착 통화 ISO 코드 (예: 'KRW', 'USD', 'EUR', 'JPY')
    """
    url = f"https://open.er-api.com/v6/latest/{from_currency}"
    try:
        data = requests.get(url, timeout=10).json()
        rate = data["rates"].get(to_currency)
        if rate is None:
            return ExchangeRateInfo(from_currency=from_currency, to_currency=to_currency,
                                    rate=0, error=f"'{to_currency}' 통화 미지원")
        return ExchangeRateInfo(from_currency=from_currency, to_currency=to_currency,
                                rate=rate, last_updated=data.get("time_last_update_utc", "")[:25])
    except Exception as e:
        return ExchangeRateInfo(from_currency=from_currency, to_currency=to_currency,
                                rate=0, error=f"API 호출 실패: {e}")

print("✅ 진짜 환율 도구 준비 (Pydantic 반환)")

✅ 진짜 환율 도구 준비 (Pydantic 반환)


**👀 코드 구조**

| 부분 | 역할 |
| --- | --- |
| Pydantic 스키마 | 반환값을 정형화 (본 노트북 패턴) |
| `requests.get(...)` | 외부 API 호출 |
| `try/except` | 네트워크 실패 처리 |
| `Optional[str] error` | 실패 시 메시지를 같은 객체에 담음 |

## 2-2. LLM과 연결


In [7]:
# 다양한 질문 — LLM이 통화 코드 자동 매핑
for q in [
    "100달러를 한국 돈으로 환산해줘",
    "1만 엔이면 한국 돈 얼마야?",
    "500유로를 달러로 바꾸면?",
    "5000원이면 일본 엔으로 얼마지?",
]:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [get_exchange_rate])}")
    print()

❓ 100달러를 한국 돈으로 환산해줘
💬 현재 환율에 따르면 1달러는 약 1,347.20원입니다. 따라서 100달러는 약 134,720원이 됩니다.

❓ 1만 엔이면 한국 돈 얼마야?
💬 현재 환율에 따르면 1 일본 엔(JPY)은 약 8.72 한국 원(KRW)입니다. 따라서 1만 엔은 약 87,176 원입니다.

❓ 500유로를 달러로 바꾸면?
💬 현재 환율에 따르면 1 유로는 약 1.1552 달러입니다. 따라서 500 유로를 달러로 바꾸면 약 577.58 달러가 됩니다.

❓ 5000원이면 일본 엔으로 얼마지?
💬 현재 환율에 따르면 1원이 약 0.11471 일본 엔입니다. 따라서 5000원은 약 573.55 일본 엔입니다.



**👀 관찰 포인트**

- "달러", "엔", "유로", "원" → LLM이 **ISO 코드**로 자동 변환
- 가짜 도구와 달리 **실제 시장 환율**이 반영됨
- 반환값이 `ExchangeRateInfo` Pydantic 객체 → 코드 안정성 ↑

> 💡 도구 정의는 가짜 버전과 거의 같지만 **데이터의 가치가 완전히 달라졌어요**.


---

# Step 3. ② 위키피디아 검색 → `WikiInfo`

LLM이 모르는 **최신 정보·구체적 사실**을 위키피디아에서 검색.

## 3-1. API 미리보기


In [8]:
import requests
from urllib.parse import quote

title = "랭체인"
encoded_title = quote(title)
url = f"https://ko.wikipedia.org/api/rest_v1/page/summary/{encoded_title}"

# ⚠️ User-Agent 필수
headers = {
    "User-Agent": "TextAnalysisCourse/1.0 (educational use)"
}

response = requests.get(url, headers=headers, timeout=10)

# 상태 코드 먼저 확인 (디버깅 습관)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(f"📖 제목: {data.get('title')}")
    print(f"📝 설명: {data.get('description')}")
    print(f"📄 요약:\n{data.get('extract', '')[:200]}...")
else:
    print(f"❌ 요청 실패: {response.text[:200]}")

Status: 200
📖 제목: 랭체인
📝 설명: LLM을 사용하여 애플리케이션 생성을 단순화하도록 설계된 프레임워크
📄 요약:
랭체인(LangChain)은 애플리케이션에 대형 언어 모델(LLM)의 통합을 돕는 소프트웨어 프레임워크이다. 언어 모델 통합 프레임워크로서 랭체인의 사용 사례는 문서 분석 및 자동 요약, 챗봇, 코드 분석 등 일반적인 언어 모델의 사용 사례와 크게 겹친다....


## 3-2. Pydantic 스키마 + 도구 정의


In [9]:
class WikiInfo(BaseModel):
    title: str = Field(description="문서 제목")
    description: Optional[str] = Field(default=None, description="짧은 설명")
    summary: str = Field(description="본문 요약")
    found: bool = Field(description="검색 성공 여부")

@tool
def search_wikipedia(query: str) -> WikiInfo:
    """한국어 위키피디아에서 항목을 검색해 요약을 반환합니다.

    인물·기업·개념·사건·역사 등 사실 정보 질문에 사용하세요.

    Args:
        query: 검색어 (예: '아인슈타인', '랭체인', '광주민주화운동')
    """
    url = f"https://ko.wikipedia.org/api/rest_v1/page/summary/{query}"
    try:
        data = requests.get(url, timeout=10).json()
        if "extract" not in data:
            return WikiInfo(title=query, summary="", found=False)
        return WikiInfo(title=data.get("title", query),
                        description=data.get("description"),
                        summary=data.get("extract", ""), found=True)
    except Exception as e:
        return WikiInfo(title=query, summary=f"API 호출 실패: {e}", found=False)

print("✅ 위키피디아 도구 준비 (Pydantic 반환)")

✅ 위키피디아 도구 준비 (Pydantic 반환)


In [10]:
# 다양한 분야의 질문
for q in [
    "아인슈타인은 누구야?",
    "광합성이 뭐야? 간단히 설명해줘",
    "조선 세종대왕에 대해 알려줘",
]:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [search_wikipedia])[:200]}...")
    print()

❓ 아인슈타인은 누구야?
💬 ...

❓ 광합성이 뭐야? 간단히 설명해줘
💬 광합성은 식물, 조류, 일부 박테리아가 태양의 빛 에너지를 이용하여 이산화탄소와 물로부터 유기 화합물(주로 포도당)을 합성하는 과정입니다. 이 과정에서 산소가 방출되며, 이는 지구의 생명체에게 필수적인 산소 공급원입니다. 광합성은 두 가지 주요 단계로 나눌 수 있습니다: 빛 반응과 어두운 반응(칼빈 회로). 이 과정은 지구 생태계의 기초를 형성하며, 에너지...

❓ 조선 세종대왕에 대해 알려줘
💬 ...



**👀 관찰 포인트**

- LLM이 질문에서 **검색어를 추출** ("아인슈타인은 누구야?" → query="아인슈타인")
- 검색 실패도 `found=False` 로 명확하게 표현 (본 노트북의 `ProductInfo` 패턴 동일)
- 위키 응답을 받아 **자연스러운 한국어로 재구성**

이게 RAG(검색 증강 생성)의 가장 단순한 형태예요. Day 4-5에서 본격적으로 배웁니다.


---

# Step 4. ③ 날씨 API → `WeatherInfo`

Open-Meteo는 **좌표(위도·경도)** 를 받아 날씨를 반환합니다. 도시명 매핑 dict를 함께 두면 깔끔.

## 4-1. API 미리보기


In [11]:
# 서울 좌표로 날씨 조회
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 37.5665, "longitude": 126.9780,
    "current": "temperature_2m,relative_humidity_2m,weather_code",
}
data = requests.get(url, params=params, timeout=10).json()
print(data["current"])

{'time': '2026-09-15T01:15', 'interval': 900, 'temperature_2m': 21.0, 'relative_humidity_2m': 64, 'weather_code': 0}


**`weather_code` 의미** (WMO 표준)

| 코드 | 의미 | 코드 | 의미 |
| --- | --- | --- | --- |
| 0 | 맑음 ☀️ | 51-67 | 비 🌧️ |
| 1-3 | 부분 흐림 ⛅ | 71-77 | 눈 ❄️ |
| 45-48 | 안개 🌫️ | 80-99 | 소나기·뇌우 ⛈️ |

## 4-2. 도시 좌표 사전 + 날씨 설명


In [12]:
# 주요 도시 좌표
CITIES = {
    "서울":   (37.5665, 126.9780),
    "부산":   (35.1796, 129.0756),
    "제주":   (33.4996, 126.5312),
    "도쿄":   (35.6762, 139.6503),
    "오사카": (34.6937, 135.5023),
    "뉴욕":   (40.7128, -74.0060),
    "파리":   (48.8566, 2.3522),
    "런던":   (51.5074, -0.1278),
}

WEATHER_DESC = {
    0: "맑음", 1: "대체로 맑음", 2: "부분 흐림", 3: "흐림",
    45: "안개", 51: "약한 이슬비", 61: "약한 비", 63: "보통 비",
    65: "강한 비", 71: "약한 눈", 75: "강한 눈", 80: "소나기", 95: "뇌우",
}
print(f"✅ {len(CITIES)}개 도시 좌표 준비")

✅ 8개 도시 좌표 준비


## 4-3. Pydantic 스키마 + 도구 정의


In [13]:
# 정형 반환 스키마
class WeatherInfo(BaseModel):
    city: str = Field(description="도시 이름")
    temperature_c: Optional[float] = Field(default=None, description="섭씨 온도")
    humidity_percent: Optional[int] = Field(default=None, description="습도 %")
    condition: str = Field(description="날씨 상태 (맑음/흐림/비 등)")
    error: Optional[str] = Field(default=None, description="오류 메시지")

print("✅ WeatherInfo 스키마 준비")

✅ WeatherInfo 스키마 준비


In [14]:
@tool
def get_weather(city: str) -> WeatherInfo:
    """도시의 현재 날씨를 조회합니다.

    Args:
        city: 한국어 도시 이름 ('서울', '부산', '제주', '도쿄', '오사카', '뉴욕', '파리', '런던' 중 하나)
    """
    if city not in CITIES:
        return WeatherInfo(city=city, condition="알 수 없음",
                           error=f"미지원 도시. 가능: {list(CITIES.keys())}")
    lat, lon = CITIES[city]
    params = {"latitude": lat, "longitude": lon,
              "current": "temperature_2m,relative_humidity_2m,weather_code"}
    try:
        data = requests.get("https://api.open-meteo.com/v1/forecast",
                            params=params, timeout=10).json()
        c = data["current"]
        return WeatherInfo(city=city, temperature_c=c["temperature_2m"],
                           humidity_percent=c["relative_humidity_2m"],
                           condition=WEATHER_DESC.get(c["weather_code"], "알 수 없음"))
    except Exception as e:
        return WeatherInfo(city=city, condition="알 수 없음", error=f"API 호출 실패: {e}")

print("✅ 날씨 도구 준비 (Pydantic 반환)")

✅ 날씨 도구 준비 (Pydantic 반환)


In [15]:
for q in [
    "서울 지금 날씨 어때?",
    "도쿄 날씨 알려줘",
    "런던하고 파리 날씨 비교해줘",   # 도구를 2번 호출!
]:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [get_weather])}")
    print()

❓ 서울 지금 날씨 어때?
💬 서울의 현재 날씨는 맑고, 기온은 21도, 습도는 64%입니다.

❓ 도쿄 날씨 알려줘
💬 도쿄의 현재 날씨는 다음과 같습니다:
- 기온: 24.4°C
- 습도: 93%
- 날씨 상태: 약한 이슬비

❓ 런던하고 파리 날씨 비교해줘
💬 현재 런던과 파리의 날씨는 다음과 같습니다:

- **런던**
  - 온도: 19.1°C
  - 습도: 83%
  - 날씨: 흐림

- **파리**
  - 온도: 20.9°C
  - 습도: 71%
  - 날씨: 맑음

파리는 런던보다 온도가 높고, 날씨가 맑습니다.



**👀 관찰 포인트**

- "런던하고 파리 날씨 비교해줘" → LLM이 **같은 도구를 두 번 호출** (런던, 파리)
- 이게 **Parallel Tool Calling**
- LLM이 두 결과를 보고 비교하는 자연어 답변까지 생성
- 모든 응답이 `WeatherInfo` Pydantic 객체로 통일 → 후처리 안정


---

# Step 5. ④ 국가 정보 API → `CountryInfo`

여행 가기 전 그 나라의 **수도·통화·인구·언어** 같은 기본 정보 조회.

## 5-1. API 미리보기


In [11]:
# 일본 정보 조회
data = requests.get("https://api.restcountries.com/countries/v5/codes.alpha_2/JP", {'headers': { 'Authorization': 'Bearer rc_live_demo' }}, timeout=10).json()
print(data)
japan = data[0]

print(f"🏳️  공식 명칭: {japan['name']['common']}")
print(f"🏛️  수도:    {japan['capital'][0]}")
print(f"👥 인구:    {japan['population']:,}명")
print(f"💱 통화:    {list(japan['currencies'].keys())[0]}")

{'errors': [{'message': 'Authorization key required.', 'code': 'authKeyMissing'}]}


KeyError: 0

## 5-2. Pydantic 스키마 + 도구 정의


In [17]:
class CountryInfo(BaseModel):
    name: str = Field(description="국가명")
    capital: Optional[str] = Field(default=None, description="수도")
    population: Optional[int] = Field(default=None, description="인구")
    currencies: list[str] = Field(default_factory=list, description="통화 목록")
    languages: list[str] = Field(default_factory=list, description="공용어 목록")
    found: bool = Field(description="조회 성공 여부")

print("✅ CountryInfo 스키마 준비")

✅ CountryInfo 스키마 준비


In [18]:
@tool
def get_country_info(country_name: str) -> CountryInfo:
    """국가의 기본 정보를 조회합니다.

    Args:
        country_name: 영어 국가 이름 (예: 'japan', 'france', 'korea', 'usa')
    """
    url = f"https://restcountries.com/v3.1/name/{country_name}"
    try:
        data = requests.get(url, timeout=10).json()
        if not isinstance(data, list) or not data:
            return CountryInfo(name=country_name, found=False)
        c = data[0]
        return CountryInfo(
            name=c["name"]["common"],
            capital=c.get("capital", [None])[0],
            population=c["population"],
            currencies=list(c.get("currencies", {}).keys()),
            languages=list(c.get("languages", {}).values()),
            found=True,
        )
    except Exception:
        return CountryInfo(name=country_name, found=False)

print("✅ 국가 정보 도구 준비 (Pydantic 반환)")

✅ 국가 정보 도구 준비 (Pydantic 반환)


In [19]:
for q in [
    "일본 수도가 어디야?",
    "프랑스 인구 알려줘",
    "베트남의 통화랑 공용어가 뭐야?",
]:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [get_country_info])}")
    print()

❓ 일본 수도가 어디야?
💬 일본의 수도는 도쿄(Tokyo)입니다.

❓ 프랑스 인구 알려줘
💬 프랑스에 대한 정보를 찾을 수 없습니다. 다른 질문이 있으시면 말씀해 주세요!

❓ 베트남의 통화랑 공용어가 뭐야?
💬 



**👀 관찰 포인트**

- LLM이 한국어 국가명을 영어로 변환 ("일본" → "japan")
- `list[str]` 필드로 다중 통화·다중 언어 처리
- 검색 실패는 `found=False` 로 명확히 (본 노트북 패턴 동일)


---

# Step 6. ⭐ 통합 미니 챗봇: 여행 도우미

4개 API 도구를 **하나의 LLM에 모두 등록**해서 진짜 쓸 만한 챗봇을 만듭니다.

## 6-1. 4개 도구 통합


In [20]:
# 4개 API 도구 모두 등록 — 반환값 모두 Pydantic
travel_tools = [
    get_exchange_rate,    # → ExchangeRateInfo
    search_wikipedia,     # → WikiInfo
    get_weather,          # → WeatherInfo
    get_country_info,     # → CountryInfo
]

print(f"✅ 여행 도우미: {len(travel_tools)}개 도구 보유")
for t in travel_tools:
    ret = t.func.__annotations__.get("return", "?")
    ret_name = getattr(ret, "__name__", str(ret))
    print(f"   🔧 {t.name:20s} → {ret_name}")

✅ 여행 도우미: 4개 도구 보유
   🔧 get_exchange_rate    → ExchangeRateInfo
   🔧 search_wikipedia     → WikiInfo
   🔧 get_weather          → WeatherInfo
   🔧 get_country_info     → CountryInfo


**👀 결과**

```
🔧 get_exchange_rate    → ExchangeRateInfo
🔧 search_wikipedia     → WikiInfo
🔧 get_weather          → WeatherInfo
🔧 get_country_info     → CountryInfo
```

**모든 도구가 Pydantic 객체를 반환**합니다. LLM 입장에서는 동일하지만 우리 코드는 정형 안정.

## 6-2. 단일 도구 호출이 필요한 질문


In [21]:
# 한 번에 도구 하나만 호출하면 되는 질문
for q in [
    "도쿄 지금 날씨 어때?",
    "일본 수도가 어디고 인구는 얼마야?",
    "100달러는 한국 돈으로 얼마야?",
    "에펠탑이 뭐야?",
]:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, travel_tools)}")
    print()

❓ 도쿄 지금 날씨 어때?
💬 현재 도쿄의 날씨는 약한 이슬비가 내리고 있으며, 기온은 24.4도, 습도는 93%입니다.

❓ 일본 수도가 어디고 인구는 얼마야?
💬 

❓ 100달러는 한국 돈으로 얼마야?
💬 현재 환율에 따르면 1달러는 약 1,347.20원입니다. 따라서 100달러는 약 134,720원이 됩니다.

❓ 에펠탑이 뭐야?
💬 에펠탑은 프랑스 파리의 상징적인 구조물로, 1887년부터 1889년까지 건설되었습니다. 이 탑은 철강 구조물로 이루어져 있으며, 높이는 약 300미터에 달합니다. 에펠탑은 구스타브 에펠의 설계로, 1889년 세계 박람회를 기념하기 위해 세워졌습니다. 현재는 파리의 주요 관광 명소 중 하나로, 매년 수많은 관광객들이 방문합니다. 에펠탑은 그 독특한 디자인과 역사적 중요성으로 인해 세계문화유산으로도 인정받고 있습니다.



## 6-3. 여러 도구를 동시 호출하는 복합 질문


In [22]:
# Parallel Tool Calling — 한 질문에 여러 도구 동시 호출
complex_q = "일본 여행 가려고 해. 도쿄 날씨, 일본 수도, 그리고 1000엔이 한국 돈으로 얼마인지 알려줘"
print(f"❓ {complex_q}\n")
print(f"💬 {run_with_tools(complex_q, travel_tools)}")

❓ 일본 여행 가려고 해. 도쿄 날씨, 일본 수도, 그리고 1000엔이 한국 돈으로 얼마인지 알려줘

💬 - **도쿄 날씨**: 현재 기온은 24.4도이며, 습도는 93%로 약한 이슬비가 내리고 있습니다.
- **일본의 수도**: 일본의 수도는 도쿄입니다.
- **환율**: 1000엔(JPY)은 약 8,718원(KRW)입니다.


**👀 관찰 포인트 — 가장 중요**

한 질문에 **3개 도구를 한꺼번에** 호출했어요.

```
"일본 여행 가려고 해. 도쿄 날씨, 일본 수도, 1000엔이 한국 돈으로?"
         ↓
[자동] get_weather(city="도쿄")           → WeatherInfo
[자동] get_country_info(country_name="japan")  → CountryInfo
[자동] get_exchange_rate(JPY, KRW)        → ExchangeRateInfo
         ↓
LLM이 3개 Pydantic 결과를 종합해 한 답변으로 정리
```

이게 **Parallel Tool Calling**의 진짜 매력. 모든 결과가 정형 객체라 LLM이 종합하기도 안정적입니다.

> 💡 다음 5교시(ReAct Agent)에서는 한 단계 더 나아가, 도구 결과를 보고 **추가 도구를 또 호출**하는 다단계 추론을 배워요.


---

# Step 7. 🎯 도전 과제

본인이 관심 있는 무료 API를 골라 **Pydantic 정형 반환** 도구를 만들어보세요.

## 과제 A: 좌표 자동 변환 (실무 빈도 ★★★★☆)

Step 4 의 `CITIES` 사전을 없애고, **Open-Meteo Geocoding API** 로 도시명 → 좌표 자동 변환.

```python
class GeoLocation(BaseModel):
    city: str
    latitude: float
    longitude: float
    country: Optional[str]

@tool
def get_coordinates(city: str) -> GeoLocation:
    """도시 이름을 위경도 좌표로 변환합니다."""
    url = "https://geocoding-api.open-meteo.com/v1/search"
    data = requests.get(url, params={"name": city, "language": "ko"}, timeout=10).json()
    r = data["results"][0]
    return GeoLocation(city=city, latitude=r["latitude"],
                       longitude=r["longitude"], country=r.get("country"))
```

이제 "마닐라 날씨", "이스탄불 날씨" 다 됩니다.

---

## 과제 B: 캐싱으로 API 호출 줄이기 (실무 빈도 ★★★★☆)

환율은 분 단위로 안 변하니까 15분 캐싱.

```python
import time

class CachedExchangeRate(BaseModel):
    from_currency: str
    to_currency: str
    rate: float
    cached: bool

_cache = {}
_CACHE_TTL = 15 * 60

@tool
def get_exchange_rate_cached(from_currency: str, to_currency: str) -> CachedExchangeRate:
    """환율 조회 (15분 캐싱)."""
    key = f"{from_currency}_{to_currency}"
    now = time.time()
    if key in _cache and now - _cache[key]["t"] < _CACHE_TTL:
        return CachedExchangeRate(**_cache[key]["data"], cached=True)
    # 실제 API 호출
    ...
```

실무 비용·속도 개선의 기본 패턴.

---

## 과제 C: 한국 공공 API (실무 빈도 ★★★★★)

[공공데이터포털](https://www.data.go.kr/)에서 API를 골라 도구화.

예시:
- 미세먼지·대기질 → `AirQualityInfo` 스키마
- 지하철 도착 → `SubwayArrival` 스키마
- 기상청 단기예보 → `KMAForecast` 스키마

> ⚠️ 대부분 API 키 필요. 환경변수에 저장하는 습관을.

---

## 과제 D: 본인이 좋아하는 무료 API 추가

[Public APIs 목록](https://github.com/public-apis/public-apis)에서 골라 도구화.

추천:
- 음식·영양 (Open Food Facts)
- 영화·도서 (TMDB, Open Library)
- 우주·과학 (NASA APOD)
- 게임 (PokeAPI, RAWG)

**핵심**: 반환값을 반드시 Pydantic 클래스로 정형화.

---

> 💡 **하나만 골라서 깊이.**


---

# 🎓 외부 API 실습 마무리

## 오늘 만든 4개 진짜 도구

| 도구 | API | Pydantic 반환 |
| --- | --- | --- |
| `get_exchange_rate` | ExchangeRate-API | `ExchangeRateInfo` |
| `search_wikipedia` | Wikipedia REST | `WikiInfo` |
| `get_weather` | Open-Meteo | `WeatherInfo` |
| `get_country_info` | REST Countries | `CountryInfo` |

## OutputParser·Pydantic 일관 사용 (1~4교시 정리)

| 교시 | Pydantic 사용처 |
| --- | --- |
| 1교시 | LLM 답변 정형화 (`PydanticOutputParser`) |
| 2교시 | 함수 반환값 정형화 (`AlertAction`, `LogRecord`) |
| 4교시 본 | 도구 반환값 정형화 (`ExchangeRateResult`, `TimeInfo`, `ProductInfo`) |
| 4교시 심화 | **외부 API 도구 반환값 정형화** ⭐ |

LLM 출력이든, 함수 결과든, 도구 반환이든, **외부 API 응답이든** 일관되게 Pydantic으로 받으면 전체 시스템이 안정적입니다.

## 가짜 → 진짜 변환 패턴

```python
# 본 노트북 (가짜)
class ExchangeRateResult(BaseModel): ...

@tool
def get_exchange_rate(...) -> ExchangeRateResult:
    rates = {("USD","KRW"): 1300.0, ...}
    return ExchangeRateResult(...)

# 심화 노트북 (진짜)
class ExchangeRateInfo(BaseModel): ...   # 필드만 살짝 확장

@tool
def get_exchange_rate(...) -> ExchangeRateInfo:
    data = requests.get(API_URL, timeout=10).json()
    return ExchangeRateInfo(...)
```

**Pydantic 스키마·반환 구조는 거의 똑같습니다.** 함수 안의 데이터 소스만 dict → API로 바뀐 거예요. LLM 입장에서는 가짜인지 진짜인지 모릅니다.

## 실무 체크리스트

외부 API 도구 만들 때 5가지.

- [ ] **Pydantic 스키마로 반환 정형화** ⭐
- [ ] `try/except` 로 네트워크 실패 처리
- [ ] `timeout` 명시 (5-10초 권장)
- [ ] 실패 시 `error` 필드나 `found=False` 로 명확히 표현
- [ ] docstring에 **인자 예시** 명확히 (예: 'USD', 'KRW')

## 다음 시간 예고

5교시는 **ReAct Agent** 입니다. 오늘 만든 4단계 흐름을 **자동화**하고, 도구 결과를 보고 **추가 도구를 또 호출**하는 다단계 추론까지 자동 처리.

```python
# 사용자: "일본 여행 계획 짜줘. 도쿄 날씨 보고, 비 오면 박물관 위주로"
# Agent가 자동으로:
# 1. get_weather("도쿄") → WeatherInfo 확인
# 2. 결과(비/맑음)에 따라 search_wikipedia(주제) 분기 호출
# 3. 최종 추천 생성
```

오늘 만든 Pydantic 반환 도구들이 5교시 Agent에서 그대로 동작합니다.

수고하셨습니다! 🎉
